DAY 5 ANN Implementation

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent

sys.path.insert(0, str(ROOT_DIR))

In [ ]:
from src.path_setup import MODEL_DIR
print(MODEL_DIR)

In [ ]:
from src.ann_model import build_ann, get_early_stopping
from src.data_extraction import X_train,y_train,X_test,y_test
from src.preprocessing import make_preprocessor
from src.path_setup import MODEL_DIR
import os
preprocessor = make_preprocessor(X_train)

X_train_preprocessed = preprocessor.fit_transform(X_train)



model = build_ann(input_dim=X_train_preprocessed.shape[1])

model.summary()

history = model.fit(
    X_train_preprocessed,
    y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[get_early_stopping()],
    verbose=1
)
model_path = os.path.join("../models","churn_day5_ann.keras")
model.save(model_path)


In [22]:
from tensorflow import keras
from src.data_extraction import X_test
model = keras.models.load_model(
    "../models/churn_day5_ann.keras"
)

model.summary()

X_test_processed = preprocessor.transform(X_test)
# Predicted probabilities
y_prob = model.predict(X_test_processed)

# Convert probabilities into binary predictions
y_pred = (y_prob >= 0.5).astype(int).ravel()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_27 (Dense)                │ (None, 64)             │         2,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,173 (59.27 KB)

 Trainable params: 5,057 (19.75 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 10,116 (39.52 KB)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


In [30]:
import pandas as pd
from sklearn.metrics import (
    classification_report,
  
)
from src.data_extraction import y_test


report = classification_report(
    y_test,
    y_pred,
    target_names=["No Churn", "Churn"],
    output_dict=True
)

report_df = pd.DataFrame(report).transpose()
report_df.to_csv("../reports/day_5_ann_results.csv")

print("Evaluation results saved successfully!")

Evaluation results saved successfully!
